# Class lesson 01 - PySpark
Objectives: 
- Setup Databricks
- Create a new notebook for todays PySpark code-along in class
- Success???

---
- SQL notebook
- Python notebook

Choosing either of them makes all cells in notebook to that choice(Either .ipynb file or SQL notebook)
---
Click ENVIRONMENT button -> Choose 32gb(will makes things run smoother and faster instead of just using 16gb)
Regarding running SQL queries or Python code in a Python/SQL notebook, you should use the magic(function(?)) %python or %sql

In [0]:
DATA_PATH = "/Volumes/data/olympic_games/raw_data/" # This is the path to the data

df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True) # Read the data
df_regions = spark.read.csv(f"{DATA_PATH}/noc_regions.csv", header=True) # Read the data

df_athletes # Display the data
df_regions # Display the data



In [0]:
df_athletes.limit(2).display() # Display the first two rows

In [0]:
df_athletes.printSchema() # Print the schema


In [0]:
(df_athletes.count(), len(df_athletes.columns)) # Count the number of rows and columns

# Infer my schema
- Note: Sex, Age, Height Weight, Year becomes string when trying to infer the schema. It is because it contains null values.

In [0]:
df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, inferSchema=True) # Read the data

df_athletes


In [0]:
display(df_athletes) # Display the data

## Define an explicit schema

In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_athletes_schema)

## EDA
- Eda on this dataset

In [0]:
# Time to do EDA
df_athletes_schema.groupBy("NOC", "Medal").count().filter(
    "NOC IN ('SWE', 'NOR', 'FIN', 'DEN', 'ISL') AND Medal != 'NA'"
).sort("NOC", "Medal").display() 

## Spark SQL
- Time to do some work with Spark SQL

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

# TODO: pick out swedish medals in table tennis
spark.sql("""
          SELECT
            DISTINCT sport
          FROM df_athletes_schema
          
          """).display()

In [0]:
spark.sql(
    """
          SELECT
            name, 
            sport,
            event,
            medal,
            year
          FROM df_athletes_schema
          WHERE 
            sport = 'Table Tennis' AND
            medal != 'NA' AND
            noc = 'SWE'
          """
).display()

In [0]:
df_swe_medals = spark.sql("""
          SELECT
            sport, count(medal) AS medal_count
          FROM df_athletes_schema
          WHERE noc = 'SWE' AND medal IN ('Bronze', 'Silver','Gold')
          GROUP BY sport 
          ORDER BY medal_count DESC
          """)

df_swe_medals.display() # 

In [0]:

fig = df_swe_medals.plot(kind = "bar", y="sport", x = "medal_count") # plotting a figure
fig.update_layout(yaxis = {"autorange": "reversed"}) # reversing the y-axis

## Ingesting data to Unity Catalog

In [0]:
%sql
SHOW SCHEMAS IN DATA;

CREATE TABLE IF NOT EXISTS data.olympic_games.swedish_medals AS
(
    SELECT
        name,
        age,
        height,
        weight,
        year,
        sport,
        medal
    FROM df_athletes_schema
    WHERE noc = 'SWE' and medal IN ('Bronze', 'Silver', 'Gold')

)

In [0]:
%sql

SELECT
    name,
    sport,
    medal,
    year
FROM data.olympic_games.swedish_medals
WHERE medal = 'Gold'
ORDER BY year ASC;